# Практикум: модули, пакеты и структура проекта

Python, 9 класс (углублённый). Раздел 3.1, пара 1.

Двенадцать задач тремя уровнями:

- **Уровень 1 (задачи 1–6)** — обязательный. По одной идее на задачу.
- **Уровень 2 (задачи 7–10)** — на сочетание идей.
- **Уровень 3 (задачи 11–12)** — со звёздочкой.

В большинстве задач вы пишете не код в ячейке, а **файлы проекта**. В ячейке есть словарь `files` —
«путь → текст файла», как в лекции. Текст файла пишется внутри тройных кавычек `'''...'''`; отступ слева
можно оставить любым, лишний уберётся сам. Пишите на месте `# ваш код здесь` и запускайте ячейку.
Ниже строки `# ---------- проверка ----------` — тесты: они создают из словаря папку проекта,
импортируют модули, запускают программы командой `python` и проверяют, что получилось. Если все проверки
проходят, внизу печатается «ок».

Посмотреть, что печатает ваша программа, можно в отдельной ячейке: `make_project(files)`, потом
`print(run("main.py").stdout)`. Сообщение об ошибке покажет `print(run("main.py").stderr)`.
Для этого в первой ячейке есть `make_project`, `run` и другие инструменты — эту ячейку надо запустить.

In [ ]:
# готовые инструменты для проверок — писать их не нужно, просто запустите ячейку
import ast
import builtins
import contextlib
import importlib
import io
import itertools
import os
import random
import shutil
import statistics          # стандартный модуль загружаем заранее: в задаче 4 есть свой statistics.py
import subprocess
import sys
import tempfile
import textwrap
import types
from pathlib import Path

sys.dont_write_bytecode = True        # чтобы после правки файла Python не взял его старую версию
ROOT = Path(tempfile.gettempdir()) / "py9_15_practice" / "project"


def _from_project(module):
    # загружен ли модуль из папки проекта
    try:
        root = ROOT.resolve()
        places = [getattr(module, "__file__", None)] + list(getattr(module, "__path__", None) or [])
        return any(place and Path(place).resolve().is_relative_to(root) for place in places)
    except Exception:
        return False


def make_project(files):
    """Создаёт папку проекта по словарю «путь -> текст файла». Прошлая версия стирается."""
    for name, module in list(sys.modules.items()):
        if _from_project(module):
            del sys.modules[name]
    if ROOT.exists():
        shutil.rmtree(ROOT)
    ROOT.mkdir(parents=True)
    for name, text in files.items():
        path = ROOT / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(text).lstrip("\n"), encoding="utf-8")
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    importlib.invalidate_caches()


def run(*args):
    """Запускает в папке проекта  python args...  Результат: .stdout, .stderr, .returncode.
    Посмотреть, что напечатала программа: print(run("main.py").stdout)."""
    env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "PYTHONIOENCODING": "utf-8",
           "PYTHON_COLORS": "0", "NO_COLOR": "1"}
    return subprocess.run([sys.executable, *args], cwd=ROOT, env=env, capture_output=True,
                          text=True, encoding="utf-8", timeout=60)


def error_line(result):
    """Последняя строка сообщения об ошибке программы, запущенной через run (или пустая строка)."""
    lines = result.stderr.strip().splitlines()
    return lines[-1] if lines else ""


def import_quietly(name):
    """Импортирует модуль проекта. Возвращает пару: модуль и то, что он напечатал при импорте."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        module = importlib.import_module(name)
    return module, buffer.getvalue()


def error_of(func, *args, **kwargs):
    """Вызывает func и возвращает исключение, если оно случилось, иначе None."""
    try:
        func(*args, **kwargs)
    except Exception as error:
        return error
    return None


def module_name(path):
    """'school/marks.py' -> 'school.marks', 'school/__init__.py' -> 'school'."""
    parts = list(Path(path).with_suffix("").parts)
    if parts[-1] == "__init__":
        parts.pop()
    return ".".join(parts)


def _imports(path, text):
    # пары (откуда, что): для import x — ("x", None), для from a import b — ("a", "b")
    name = module_name(path)
    package = name.split(".") if path.endswith("__init__.py") else name.split(".")[:-1]
    result = []
    for node in ast.walk(ast.parse(textwrap.dedent(text))):
        if isinstance(node, ast.Import):
            result.extend((alias.name, None) for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            base = package[:len(package) - node.level + 1] if node.level else []
            target = ".".join(base + (node.module.split(".") if node.module else []))
            result.extend((target, alias.name) for alias in node.names)
    return result


def imports_in(path, text):
    """Какие модули импортирует файл path с текстом text — и в начале, и внутри функций."""
    found = set()
    for target, name in _imports(path, text):
        found.add(target)
        if name:
            found.add(f"{target}.{name}")
    return found


def import_cycle(files):
    """Ищет круг в импортах между модулями проекта: список модулей круга или None."""
    sources = {module_name(path): (path, text) for path, text in files.items() if path.endswith(".py")}
    graph = {}
    for module, (path, text) in sources.items():
        edges = set()
        for target, name in _imports(path, text):
            if name and f"{target}.{name}" in sources:
                edges.add(f"{target}.{name}")
            elif target in sources:
                edges.add(target)
        graph[module] = sorted(edges)
    state, stack = {}, []

    def visit(module):
        state[module] = "в пути"
        stack.append(module)
        for other in graph[module]:
            if state.get(other) == "в пути":
                return stack[stack.index(other):] + [other]
            if other not in state:
                found = visit(other)
                if found:
                    return found
        stack.pop()
        state[module] = "готов"
        return None

    for module in sorted(graph):
        if module not in state:
            found = visit(module)
            if found:
                return found
    return None


print("инструменты для проверок загружены")

## Уровень 1. Обязательный

### Задача 1. Свой модуль

Ниже — две функции для слов. Перенесите их в модуль `words.py`, а в первой строке файла напишите
докстроку модуля. В `main.py` напишите программу, которая импортирует модуль и для слов `Шалаш`, `кот`
и `Анна` печатает:

```
Шалаш: гласных 2, палиндром
кот: гласных 1, не палиндром
Анна: гласных 2, палиндром
```

```python
VOWELS = "аеёиоуыэюя"


def count_vowels(word):
    count = 0
    for letter in word.lower():
        if letter in VOWELS:
            count += 1
    return count


def is_palindrome(word):
    word = word.lower()
    return word == word[::-1]
```

*Подсказка:* докстрока модуля — строка в тройных кавычках `"""..."""` в самом начале файла, как у `marks.py`
в лекции. Функции в `main.py` не копируйте, их нужно импортировать.

In [ ]:
files = {
    "words.py": '''
"""Функции для слов: сколько в слове гласных и является ли оно палиндромом."""

VOWELS = "аеёиоуыэюя"


def count_vowels(word):
    """Сколько гласных букв в слове."""
    count = 0
    for letter in word.lower():
        if letter in VOWELS:
            count += 1
    return count


def is_palindrome(word):
    """Слово читается одинаково в обе стороны?"""
    word = word.lower()
    return word == word[::-1]
''',
    "main.py": '''
import words

for word in ["Шалаш", "кот", "Анна"]:
    kind = "палиндром" if words.is_palindrome(word) else "не палиндром"
    print(f"{word}: гласных {words.count_vowels(word)}, {kind}")
''',
}


# ---------- проверка ----------
make_project(files)
words, printed = import_quietly("words")
assert printed == "", f"при импорте модуль ничего не печатает, а напечатал: {printed!r}"
assert words.__doc__ and words.__doc__.strip(), "в первой строке words.py нужна докстрока модуля"
assert hasattr(words, "count_vowels") and hasattr(words, "is_palindrome"), "функции должны лежать в words.py"
assert words.count_vowels("Шалаш") == 2 and words.count_vowels("кот") == 1
assert words.count_vowels("") == 0 and words.count_vowels("ЁЖ") == 1
assert words.is_palindrome("Шалаш") and words.is_palindrome("Анна") and not words.is_palindrome("кот")
assert "words" in imports_in("main.py", files["main.py"]), "main.py должен импортировать модуль words"
assert "def count_vowels" not in files["main.py"] and "def is_palindrome" not in files["main.py"], \
    "функции живут в words.py, в main.py их не копируют"
r = run("main.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == "Шалаш: гласных 2, палиндром\nкот: гласных 1, не палиндром\nАнна: гласных 2, палиндром\n", \
    f"main.py напечатал:\n{r.stdout}"
print("Задача 1: ок")

### Задача 2. Импорт без лишнего

Модуль `money.py` переводит рубли в копейки и обратно — как в задаче 6 КР № 1. Автор проверял функции
прямо в файле, и теперь каждый, кто пишет `import money`, получает на экран две лишние строки.
Переделайте модуль: при импорте он ничего не печатает, а при запуске `python money.py` печатает те же
две строки, что и сейчас:

```
1250 7 300
12.50 0.07 3.00
```

*Подсказка:* блок 2 лекции. Сами функции не трогайте.

In [ ]:
files = {
    "money.py": '''
"""Деньги в копейках: из строки и обратно в строку."""


def to_kopecks(text):
    """'12.5' -> 1250, '0.07' -> 7, '3' -> 300."""
    rubles, _, kopecks = text.partition(".")
    return int(rubles) * 100 + int((kopecks + "00")[:2])


def format_rub(kopecks):
    """1250 -> '12.50'."""
    return f"{kopecks // 100}.{kopecks % 100:02d}"


if __name__ == "__main__":
    print(to_kopecks("12.5"), to_kopecks("0.07"), to_kopecks("3"))
    print(format_rub(1250), format_rub(7), format_rub(300))
''',
}


# ---------- проверка ----------
make_project(files)
money, printed = import_quietly("money")
assert printed == "", f"при импорте модуль ничего не печатает, а напечатал: {printed!r}"
assert money.to_kopecks("12.5") == 1250 and money.format_rub(7) == "0.07", "функции работают как раньше"
r = run("money.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == "1250 7 300\n12.50 0.07 3.00\n", f"при запуске напечатано: {r.stdout!r}"
print("Задача 2: ок")

### Задача 3. Счётчик посещений

Модуль `visits.py` считает посещения сайта, он написан правильно — его не меняйте. А `main.py` после
трёх посещений печатает `посещений: 0`. Почините `main.py`, чтобы он печатал `посещений: 3`.

*Подсказка:* «Что выведет?» из блока 1 лекции. Число посещений должна считать не `main.py`, а модуль:
проверка подменит `visits.py` на другой и посмотрит, изменится ли ответ.

In [ ]:
files = {
    "visits.py": '''
"""Счётчик посещений сайта."""

total = 0


def visit():
    global total
    total += 1
''',
    "main.py": '''
import visits

for _ in range(3):
    visits.visit()
print("посещений:", visits.total)
''',
}


# ---------- проверка ----------
make_project(files)
r = run("main.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == "посещений: 3\n", f"main.py напечатал: {r.stdout!r}"

other_visits = '''
total = 0


def visit():
    global total
    total += 2
'''
make_project({**files, "visits.py": other_visits})       # visit прибавляет по 2
r = run("main.py")
assert r.stdout == "посещений: 6\n", "main.py должен брать число посещений из модуля, а не считать сам"
print("Задача 3: ок")

### Задача 4. Чужое имя

Ученик сложил свою функцию `average` в файл `statistics.py`, а в `main.py` хотел взять медиану
из стандартного модуля `statistics`. Программа падает. Переименуйте свой модуль в `marks_stats.py`
и почините импорты в `main.py`, чтобы программа печатала:

```
среднее: 3.83
медиана: 4.0
```

*Подсказка:* блок 3 лекции. Медиану считает стандартный `statistics.median`, `average` — ваша. Чтобы
переименовать файл, поменяйте ключ в словаре `files`.

In [ ]:
files = {
    "marks_stats.py": '''
"""Мои функции для отметок."""


def average(marks):
    return round(sum(marks) / len(marks), 2)
''',
    "main.py": '''
import statistics
from marks_stats import average

marks = [5, 3, 4, 4, 5, 2]
print("среднее:", average(marks))
print("медиана:", statistics.median(marks))
''',
}


# ---------- проверка ----------
make_project(files)
clash = [name for name in files if name.endswith(".py") and Path(name).stem in sys.stdlib_module_names]
assert not clash, f"имя файла совпадает с модулем стандартной библиотеки: {clash}"
assert "marks_stats.py" in files, "свой модуль назовите marks_stats.py"
r = run("main.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == "среднее: 3.83\nмедиана: 4.0\n", f"main.py напечатал: {r.stdout!r}"
found = imports_in("main.py", files["main.py"])
assert "statistics" in found and "marks_stats" in found, \
    "медиана — из стандартного statistics, average — из marks_stats"
print("Задача 4: ок")

### Задача 5. Лицо пакета

Пакет `textkit` работает с текстом: в `words.py` лежат `count_words` и `longest_word`, в `case.py` —
`title_case`. Оба модуля уже написаны. Напишите `textkit/__init__.py`, чтобы заработал готовый `main.py`:

- первая строка — докстрока пакета;
- наверх подняты `count_words`, `longest_word` и `title_case` — **относительным** импортом;
- внутренняя `_clean` наверх не поднимается, `import *` не используется.

*Подсказка:* блок 4 лекции, второй вариант `school/__init__.py`.

In [ ]:
files = {
    "textkit/__init__.py": '''
"""Пакет textkit: функции для работы с текстом."""

from .words import count_words, longest_word
from .case import title_case
''',
    "textkit/words.py": '''
def _clean(text):
    """Слова без знаков препинания по краям."""
    return [word.strip(".,!?;:") for word in text.split()]


def count_words(text):
    return len(_clean(text))


def longest_word(text):
    words = _clean(text)
    return max(words, key=len) if words else ""
''',
    "textkit/case.py": '''
def title_case(text):
    """Каждое слово — с заглавной буквы."""
    return " ".join(word.capitalize() for word in text.split())
''',
    "main.py": '''
from textkit import count_words, longest_word, title_case

text = "мама мыла раму, а папа читал газету."
print(count_words(text))
print(longest_word(text))
print(title_case("журнал девятого класса"))
''',
}


# ---------- проверка ----------
make_project(files)
textkit, printed = import_quietly("textkit")
assert printed == "", f"при импорте пакет ничего не печатает, а напечатал: {printed!r}"
assert textkit.__doc__ and textkit.__doc__.strip(), "в первой строке __init__.py нужна докстрока пакета"
for name in ["count_words", "longest_word", "title_case"]:
    assert hasattr(textkit, name), f"textkit.{name} не найдена — поднимите её в __init__.py"
assert textkit.count_words is textkit.words.count_words and textkit.title_case is textkit.case.title_case, \
    "функции не переписывают заново, а импортируют из модулей пакета"
assert not hasattr(textkit, "_clean"), "_clean — внутренняя функция, наверх её не поднимают"
init_tree = ast.parse(textwrap.dedent(files["textkit/__init__.py"]))
init_imports = [node for node in ast.walk(init_tree) if isinstance(node, (ast.Import, ast.ImportFrom))]
assert all(isinstance(node, ast.ImportFrom) and node.level >= 1 for node in init_imports), \
    "внутри пакета — относительный импорт: from .words import ..."
assert all(alias.name != "*" for node in init_imports for alias in node.names), "import * не используйте"
r = run("main.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == "7\nгазету\nЖурнал Девятого Класса\n", f"main.py напечатал:\n{r.stdout}"
print("Задача 5: ок")

### Задача 6. Почините пакет

Пакет `cafe` считает сумму заказа. Автор писал его, запуская `python order.py` из папки `cafe`, и всё
работало. Но из корня проекта `import cafe` падает. Почините импорты в `cafe/__init__.py`
и `cafe/order.py`.

Потом впишите в `COMMAND` команду, которая **из корня проекта** запускает пример из `cafe/order.py` —
список аргументов для `run`, как в лекции: `run("main.py")` — это `COMMAND = ["main.py"]`.

```
$ python ...ваша команда...
170
```

*Подсказка:* блок 4 лекции — относительный импорт и запуск модуля пакета.

In [ ]:
files = {
    "cafe/__init__.py": '''
"""Кафе: меню и заказы."""
from .order import total
''',
    "cafe/menu.py": '''
PRICES = {"чай": 50, "пирожок": 70, "суп": 150}
''',
    "cafe/order.py": '''
from .menu import PRICES


def total(items):
    """Сумма заказа в рублях."""
    result = 0
    for item in items:
        result += PRICES[item]
    return result


if __name__ == "__main__":
    print(total(["чай", "пирожок", "чай"]))
''',
}

COMMAND = ["-m", "cafe.order"]


# ---------- проверка ----------
make_project(files)
cafe, printed = import_quietly("cafe")
assert printed == "", f"при импорте пакет ничего не печатает, а напечатал: {printed!r}"
assert cafe.total(["чай", "пирожок", "чай"]) == 170 and cafe.total([]) == 0
assert COMMAND, "впишите команду в COMMAND"
r = run(*COMMAND)
assert r.returncode == 0, f"команда {COMMAND} падает: {error_line(r)}"
assert r.stdout == "170\n", f"команда напечатала: {r.stdout!r}"
make_project({**files, "main.py": "from cafe import total\nprint(total(['суп', 'чай']))\n"})
r = run("main.py")
assert r.returncode == 0, f"программа в корне проекта не может пользоваться пакетом: {error_line(r)}"
assert r.stdout == "200\n"
print("Задача 6: ок")

## Уровень 2. На сочетание идей

### Задача 7. Разорвать круг

Программа библиотеки разложена по двум модулям: `books.py` выдаёт книги, `readers.py` работает со списком
желаний читателя. Модули импортируют друг друга, и `main.py` падает. Разорвите круг: вынесите то, что
нужно обоим модулям, в новый модуль. Функции `can_borrow` и `wishlist_available` должны остаться в своих
модулях — ими пользуются другие программы. `main.py` должен печатать:

```
True
False
['Дюна', 'Хоббит']
```

*Подсказка:* блок 5 лекции. Перенести `import` внутрь функции не годится: круг останется, просто
спрячется. Проверка смотрит импорты и внутри функций.

In [ ]:
files = {
    "config.py": '''
"""Общие данные библиотеки: каталог книг и лимит выдачи."""

CATALOG = {"Дюна": 3, "Метро 2033": 1, "Хоббит": 2}

MAX_BOOKS = 2
''',
    "books.py": '''
"""Выдача книг читателям."""
from config import CATALOG, MAX_BOOKS


def can_borrow(title, taken):
    """Можно ли выдать книгу: она есть в наличии и у читателя меньше MAX_BOOKS книг."""
    return CATALOG.get(title, 0) > 0 and len(taken) < MAX_BOOKS
''',
    "readers.py": '''
"""Списки желаний читателей."""
from config import CATALOG


def wishlist_available(wishlist):
    """Книги из списка желаний, которые есть в наличии."""
    return [title for title in wishlist if CATALOG.get(title, 0) > 0]
''',
    "main.py": '''
from books import can_borrow
from readers import wishlist_available

print(can_borrow("Дюна", ["Хоббит"]))
print(can_borrow("Дюна", ["Хоббит", "Метро 2033"]))
print(wishlist_available(["Дюна", "Война и мир", "Хоббит"]))
''',
}


# ---------- проверка ----------
make_project(files)
cycle = import_cycle(files)
assert cycle is None, f"в импортах остался круг: {' -> '.join(cycle)}"
r = run("main.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == "True\nFalse\n['Дюна', 'Хоббит']\n", f"main.py напечатал:\n{r.stdout}"
for first in ["books", "readers"]:
    r = run("-c", f"import {first}")
    assert r.returncode == 0, f"import {first} первым падает: {error_line(r)}"
books, _ = import_quietly("books")
readers, _ = import_quietly("readers")
assert books.can_borrow("Хоббит", []) and not books.can_borrow("Война и мир", [])
assert not books.can_borrow("Хоббит", ["а", "б", "в"])
assert readers.wishlist_available(["Метро 2033", "Азбука"]) == ["Метро 2033"]
print("Задача 7: ок")

### Задача 8. Из одного файла — пакет

Программа `olympiad.py` разбирает результаты олимпиады и печатает сводку. Всё в одном файле. Разложите
её по пакету `olymp`:

- `olymp/config.py` — `PASS_SCORE`;
- `olymp/parse.py` — `parse_line` и `parse`;
- `olymp/stats.py` — `passed`, `by_grade` и `top`;
- `olymp/report.py` — `format_row` и `format_table`;
- `olymp/__init__.py` — докстрока пакета и публичный интерфейс: `parse`, `passed`, `by_grade`, `top`,
  `format_table`;
- `main.py` в корне проекта — `RAW` и печать сводки под `if __name__ == "__main__":`, своих функций
  разбора и подсчёта в нём нет.

Внутри пакета — относительные импорты, кругов нет. Старый `olympiad.py` уберите из словаря. Новая
программа должна печатать ровно то же, что старая.

*Подсказка:* сначала посмотрите, что печатает старая программа: `make_project(files)`,
`print(run("olympiad.py").stdout)`. Потом переносите функции по одной и следите, кто кого импортирует.

In [ ]:
files = {
    "olymp/__init__.py": '''
"""Пакет olymp: разбор результатов олимпиады и сводка по ним."""

from .parse import parse
from .stats import passed, by_grade, top
from .report import format_table
''',
    "olymp/config.py": '''
"""Настройки олимпиады."""

PASS_SCORE = 60
''',
    "olymp/parse.py": '''
"""Разбор строк с результатами участников."""


def parse_line(line):
    """'Иванова;9;87' -> ('Иванова', 9, 87)."""
    name, grade, score = line.split(";")
    return name, int(grade), int(score)


def parse(raw):
    """Список строк -> список кортежей (фамилия, класс, балл)."""
    return [parse_line(line) for line in raw.splitlines() if line.strip()]
''',
    "olymp/stats.py": '''
"""Подсчёты: кто прошёл, распределение по классам, лучшие."""
from .config import PASS_SCORE


def passed(rows):
    """Участники, набравшие проходной балл."""
    return [row for row in rows if row[2] >= PASS_SCORE]


def by_grade(rows):
    """Фамилии участников по классам."""
    groups = {}
    for name, grade, score in rows:
        if grade not in groups:
            groups[grade] = []
        groups[grade].append(name)
    return groups


def top(rows, k):
    """k лучших: по убыванию баллов, при равенстве — по фамилии."""
    return sorted(rows, key=lambda row: (-row[2], row[0]))[:k]
''',
    "olymp/report.py": '''
"""Печать таблицы результатов."""


def format_row(row):
    """Одна строка таблицы: фамилия, класс, балл."""
    name, grade, score = row
    return f"{name:<10} {grade:>2} {score:>3}"


def format_table(rows):
    """Все строки через перевод строки."""
    return "\\n".join(format_row(row) for row in rows)
''',
    "main.py": '''
from olymp import parse, passed, by_grade, top, format_table

RAW = """Иванова;9;87
Петров;10;64
Сидоров;9;87
Кузнецова;11;92
Смирнов;10;71
Попова;9;55
Волков;11;48"""

if __name__ == "__main__":
    rows = parse(RAW)
    print("Прошли:", len(passed(rows)), "из", len(rows))
    for grade, names in sorted(by_grade(rows).items()):
        print(f"{grade} класс:", ", ".join(names))
    print("Лучшие трое:")
    print(format_table(top(rows, 3)))
''',
}


# ---------- проверка ----------
EXPECTED = (
    "Прошли: 5 из 7\n"
    "9 класс: Иванова, Сидоров, Попова\n"
    "10 класс: Петров, Смирнов\n"
    "11 класс: Кузнецова, Волков\n"
    "Лучшие трое:\n"
    "Кузнецова  11  92\n"
    "Иванова     9  87\n"
    "Сидоров     9  87\n"
)
needed = ["olymp/__init__.py", "olymp/config.py", "olymp/parse.py", "olymp/stats.py", "olymp/report.py", "main.py"]
missing = [name for name in needed if name not in files]
assert not missing, f"не хватает файлов: {missing}"
assert "olympiad.py" not in files, "старый olympiad.py больше не нужен — уберите его из словаря"
cycle = import_cycle(files)
assert cycle is None, f"в импортах круг: {' -> '.join(cycle)}"
make_project(files)
olymp, printed = import_quietly("olymp")
assert printed == "", f"при импорте пакет ничего не печатает, а напечатал: {printed!r}"
assert olymp.__doc__ and olymp.__doc__.strip(), "в первой строке olymp/__init__.py нужна докстрока"
places = {"PASS_SCORE": "config", "parse_line": "parse", "parse": "parse", "passed": "stats",
          "by_grade": "stats", "top": "stats", "format_row": "report", "format_table": "report"}
for name, place in places.items():
    module, _ = import_quietly(f"olymp.{place}")
    assert hasattr(module, name), f"{name} должна лежать в olymp/{place}.py"
    if name in ["parse", "passed", "by_grade", "top", "format_table"]:
        assert getattr(olymp, name, None) is getattr(module, name), f"поднимите {name} в olymp/__init__.py"
main_tree = ast.parse(textwrap.dedent(files["main.py"]))
own = [node.name for node in ast.walk(main_tree) if isinstance(node, ast.FunctionDef) and node.name != "main"]
assert not own, f"функции {own} живут в пакете, в main.py их не копируют"
_, printed = import_quietly("main")
assert printed == "", "при импорте main.py ничего не печатает — печать под if __name__ == '__main__'"
r = run("main.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == EXPECTED, f"main.py напечатал:\n{r.stdout}"

make_project({**files, "olymp/config.py": "PASS_SCORE = 80\n"})    # порог поменяли в одном месте
olymp, _ = import_quietly("olymp")
assert len(olymp.passed(olymp.parse("А;9;87\nБ;9;64\nВ;9;92\n"))) == 2, \
    "порог берётся из olymp/config.py, а не из копии в другом модуле"
print("Задача 8: ок")

### Задача 9. Импорты по полочкам

Напишите свою маленькую версию `isort`: `sort_imports(lines, local)` получает список строк-импортов
и возвращает их в порядке PEP 8.

- Каждая строка — `import X`, `import X as Y` или `from X import ...`; модуль `X` — второе слово строки.
  Лишних пробелов в строках нет.
- Группы по порядку: стандартная библиотека, сторонние библиотеки, свои модули. Группа определяется
  по первой части имени модуля, до точки: `os.path` — это `os`. Свои — те, чья первая часть есть
  в множестве `local`, и все относительные импорты (модуль начинается с точки).
- Внутри группы — по имени модуля без учёта регистра; у одного модуля `import` раньше `from`; при прочих
  равных — по всей строке.
- Между группами — одна пустая строка `""`, пустые группы пропускаются. Повторы убираются. Исходный
  список не меняется.

```
sort_imports(["from school.marks import average", "import numpy as np", "import sys",
              "from pathlib import Path", "import random", "from .report import report"], {"school"})
->
["from pathlib import Path", "import random", "import sys", "",
 "import numpy as np", "",
 "from .report import report", "from school.marks import average"]
```

*Подсказка:* `sys.stdlib_module_names` — множество имён модулей стандартной библиотеки (Python 3.10
и новее). Сортировка по нескольким признакам — кортеж в `key`, как на занятии 11.

In [ ]:
import sys


def _group_of(line, local):
    """0 — стандартная библиотека, 1 — сторонние, 2 — свои (по local и относительные)."""
    module = line.split()[1]
    if module.startswith("."):
        return 2
    root = module.split(".")[0]
    if root in local:
        return 2
    if root in sys.stdlib_module_names:
        return 0
    return 1


def _sort_key(line):
    """Внутри группы: по модулю без учёта регистра, import раньше from, затем вся строка."""
    module = line.split()[1]
    kind = 0 if line.startswith("import ") else 1
    return (module.lower(), kind, line)


def sort_imports(lines: list, local: set) -> list:
    """Импорты тремя группами — стандартная библиотека, сторонние, свои — через пустую строку."""
    groups = [[], [], []]
    for line in dict.fromkeys(lines):            # повторы убираются, порядок не важен
        groups[_group_of(line, local)].append(line)
    result = []
    for group in groups:
        if group:
            if result:
                result.append("")
            result.extend(sorted(group, key=_sort_key))
    return result




example = ["from school.marks import average", "import numpy as np", "import sys",
           "from pathlib import Path", "import random", "from .report import report"]
source = list(example)
assert sort_imports(example, {"school"}) == [
    "from pathlib import Path", "import random", "import sys", "",
    "import numpy as np", "",
    "from .report import report", "from school.marks import average",
], "пример из условия"
assert example == source, "исходный список менять нельзя"
assert sort_imports([], set()) == []
assert sort_imports(["import sys"], set()) == ["import sys"], "пустые группы и лишние пустые строки не нужны"
assert sort_imports(["import pandas as pd", "import numpy"], set()) == ["import numpy", "import pandas as pd"]
assert sort_imports(["from os import path", "import os"], set()) == ["import os", "from os import path"], \
    "у одного модуля import идёт раньше from"
assert sort_imports(["import os.path", "import os", "import collections.abc"], set()) == \
    ["import collections.abc", "import os", "import os.path"], "группа — по первой части имени: os.path — это os"
assert sort_imports(["import PIL", "import numpy"], set()) == ["import numpy", "import PIL"], \
    "регистр букв при сравнении не важен"
assert sort_imports(["import sys", "import json", "import sys"], set()) == ["import json", "import sys"], \
    "повторы убираются"
assert sort_imports(["from typing import List", "from typing import Dict"], set()) == \
    ["from typing import Dict", "from typing import List"], "при равных модулях — по всей строке"
assert sort_imports(["import matplotlib.pyplot as plt", "from school import report", "import json"], {"school"}) == \
    ["import json", "", "import matplotlib.pyplot as plt", "", "from school import report"]
assert sort_imports(["import marks", "import math"], {"marks"}) == ["import math", "", "import marks"], \
    "свои модули узнаются по local"
assert sort_imports(["from .marks import average", "from . import config"], set()) == \
    ["from . import config", "from .marks import average"], "относительный импорт — всегда свой"
assert sort_imports(["import requests", "from school import report"], {"school"}) == \
    ["import requests", "", "from school import report"], "пустая группа стандартной библиотеки пропускается"

rnd = random.Random(14)
pool = ["import os", "import sys", "from pathlib import Path", "import json", "import numpy as np",
        "from numpy import array", "import requests", "import marks", "from school import report",
        "from .config import LIMIT", "import random", "from collections import Counter"]
for _ in range(300):
    lines = rnd.choices(pool, k=rnd.randint(0, 8))
    result = sort_imports(lines, {"marks", "school"})
    assert set(result) - {""} == set(lines), lines
    assert len(result) - result.count("") == len(set(lines)), "каждая строка — один раз"
    assert not result or (result[0] != "" and result[-1] != ""), "пустые строки — только между группами"
    assert "\n".join(result).count("\n\n\n") == 0, "между группами — одна пустая строка"
print("Задача 9: ок")

### Задача 10. Проект целиком

Соберите с нуля проект «Дежурства» — устройством как в блоке 6 лекции:

- `duty/roster.py` — функция `make_roster(names, days)`: словарь «день → кто дежурит». Дежурят по очереди,
  по кругу: людей меньше, чем дней, — начинаем список заново. Если `names` пуст, расписание пустое.
- `duty/__init__.py` — докстрока, наверх поднята `make_roster`.
- `data/names.txt` — имена по одному в строке (уже есть).
- `main.py` — читает имена из `data/names.txt` и печатает расписание на `пн`, `вт`, `ср`, `чт`, `пт`.
- `tests/__init__.py` — пустой; `tests/test_roster.py` — не меньше двух функций `test_...` с `assert`.
  Под `if __name__ == "__main__":` они вызываются, а в конце печатается `тесты пройдены`. Тесты должны
  ловить ошибки: проверка подменит `make_roster` неправильными версиями и убедится, что ваши тесты падают.
- `README.md` — первая строка — заголовок `# ...`; в тексте есть команды `python main.py`
  и `python -m tests.test_roster`.
- `requirements.txt` — сторонних библиотек нет, хватит комментария.

```
$ python main.py
пн: Аня
вт: Боря
ср: Вера
чт: Аня
пт: Боря
```

*Подсказка:* имена прочитает `Path("data/names.txt").read_text(encoding="utf-8").split()` — как
`read_journal` в лекции. Команды запускают из корня проекта, поэтому путь относительный. Очередь по кругу —
остаток от деления: `names[i % len(names)]`.

In [ ]:
files = {
    "README.md": '''
# Дежурства

Проект составляет расписание дежурств по классу: имена берутся из файла
data/names.txt, дежурят по очереди по кругу.

## Как запустить

Расписание на учебную неделю (из корня проекта):

    python main.py

Тесты расписания:

    python -m tests.test_roster

Сторонних библиотек проект не использует — см. requirements.txt.
''',
    "requirements.txt": '''
# Сторонние библиотеки не нужны
''',
    "duty/__init__.py": '''
"""Пакет duty: расписание дежурств."""

from .roster import make_roster
''',
    "duty/roster.py": '''
"""Кто дежурит в какой день."""


def make_roster(names, days):
    """Словарь «день → кто дежурит»: дежурят по очереди, по кругу."""
    if not names:
        return {}
    return {day: names[i % len(names)] for i, day in enumerate(days)}
''',
    "main.py": '''
from pathlib import Path

from duty import make_roster

DAYS = ["пн", "вт", "ср", "чт", "пт"]


if __name__ == "__main__":
    names = Path("data/names.txt").read_text(encoding="utf-8").split()
    for day, name in make_roster(names, DAYS).items():
        print(f"{day}: {name}")
''',
    "tests/__init__.py": "",
    "tests/test_roster.py": '''
"""Тесты расписания дежурств."""

from duty import make_roster


def test_round_robin():
    roster = make_roster(["Аня", "Боря"], ["пн", "вт", "ср"])
    assert roster == {"пн": "Аня", "вт": "Боря", "ср": "Аня"}, "дежурят по очереди по кругу"


def test_empty_inputs():
    assert make_roster([], ["пн", "вт"]) == {}, "некому дежурить — расписание пустое"
    assert make_roster(["Аня"], []) == {}, "нет дней — расписание пустое"


if __name__ == "__main__":
    test_round_robin()
    test_empty_inputs()
    print("тесты пройдены")
''',
    "data/names.txt": '''
Аня
Боря
Вера
''',
}


# ---------- проверка ----------
needed = ["README.md", "requirements.txt", "main.py", "duty/__init__.py", "duty/roster.py",
          "data/names.txt", "tests/__init__.py", "tests/test_roster.py"]
missing = [name for name in needed if name not in files]
assert not missing, f"не хватает файлов: {missing}"
make_project(files)
duty, printed = import_quietly("duty")
assert printed == "", f"при импорте пакет ничего не печатает, а напечатал: {printed!r}"
assert duty.__doc__ and duty.__doc__.strip(), "в первой строке duty/__init__.py нужна докстрока"
assert duty.make_roster is import_quietly("duty.roster")[0].make_roster, "make_roster живёт в duty/roster.py"
week = ["пн", "вт", "ср", "чт", "пт"]
assert duty.make_roster(["Аня", "Боря", "Вера"], week) == {"пн": "Аня", "вт": "Боря", "ср": "Вера", "чт": "Аня", "пт": "Боря"}
assert duty.make_roster(["Аня"], ["пн", "вт"]) == {"пн": "Аня", "вт": "Аня"}
assert duty.make_roster(["Аня", "Боря", "Вера"], ["сб"]) == {"сб": "Аня"}
assert duty.make_roster([], week) == {}, "некому дежурить — расписание пустое"
assert duty.make_roster(["Аня", "Боря"], []) == {}
r = run("main.py")
assert r.returncode == 0, error_line(r)
assert r.stdout == "пн: Аня\nвт: Боря\nср: Вера\nчт: Аня\nпт: Боря\n", f"main.py напечатал:\n{r.stdout}"
make_project({**files, "data/names.txt": "Гоша\nДаша\n"})
r = run("main.py")
assert r.stdout == "пн: Гоша\nвт: Даша\nср: Гоша\nчт: Даша\nпт: Гоша\n", "имена нужно брать из data/names.txt"

test_tree = ast.parse(textwrap.dedent(files["tests/test_roster.py"]))
tests = [node.name for node in ast.walk(test_tree) if isinstance(node, ast.FunctionDef) and node.name.startswith("test_")]
assert len(tests) >= 2, "в tests/test_roster.py нужно не меньше двух функций test_..."
make_project(files)
r = run("-m", "tests.test_roster")
assert r.returncode == 0 and r.stdout.strip().endswith("тесты пройдены"), \
    f"тесты не проходят на правильном make_roster: {error_line(r) or r.stdout}"
broken_versions = {
    "все дни — первый в списке": '''
def make_roster(names, days):
    return {day: names[0] for day in days} if names else {}
''',
    "без круга: дни после конца списка пропадают": '''
def make_roster(names, days):
    return {day: names[i] for i, day in enumerate(days) if i < len(names)}
''',
}
for what, code in broken_versions.items():
    make_project({**files, "duty/roster.py": code})
    r = run("-m", "tests.test_roster")
    assert r.returncode != 0, f"make_roster сломан ({what}), а ваши тесты этого не заметили"

readme = textwrap.dedent(files["README.md"]).strip()
assert readme.startswith("# "), "README.md начинается с заголовка: # Название"
assert "python main.py" in readme and "python -m tests.test_roster" in readme, "в README.md — как запускать"
print("Задача 10: ок")

## Уровень 3. Со звёздочкой

### Задача 11. Порядок загрузки ★

Напишите `load_order(deps)`. `deps` — словарь «модуль → список модулей проекта, которые он импортирует».
Верните список всех модулей в таком порядке, чтобы каждый стоял после всех, кого он импортирует: в таком
порядке модули можно загрузить без круга. Если подходящих порядков несколько, годится любой. Если такого
порядка нет — в импортах круг, — возбудите `ImportError` с сообщением, которое начинается со слов
`циклический импорт`.

```
load_order({"main": ["pupils"], "pupils": ["config", "marks"], "marks": ["config"], "config": []})
->  ["config", "marks", "pupils", "main"]

load_order({"main": ["pupils"], "pupils": ["marks"], "marks": ["pupils"]})
->  ImportError: циклический импорт: ...
```

*Подсказка:* модуль можно поставить в ответ, когда все, кого он импортирует, уже стоят. Повторяйте: найдите
такой модуль и поставьте. Если ни одного такого нет, а модули ещё остались, — это круг. Модуль, который
импортирует сам себя, — тоже круг.

In [ ]:
def load_order(deps: dict) -> list:
    """Порядок загрузки модулей: каждый после тех, кого он импортирует. Круг — ImportError."""
    order = []
    placed = set()
    remaining = dict(deps)
    while remaining:
        # готов тот, чьи импорты все уже стоят в ответе
        ready = [name for name in remaining if all(d in placed for d in remaining[name])]
        if not ready:
            raise ImportError("циклический импорт: " + ", ".join(sorted(remaining)))
        order.extend(ready)
        placed.update(ready)
        remaining = {name: needs for name, needs in remaining.items() if name not in placed}
    return order




def check_order(deps, order):
    # order — все модули по разу, и каждый стоит после тех, кого импортирует
    if sorted(order) != sorted(deps):
        return False
    position = {module: i for i, module in enumerate(order)}
    return all(position[other] < position[module] for module in deps for other in deps[module])


def has_order(deps):
    # есть ли подходящий порядок — честным перебором всех перестановок
    return any(check_order(deps, list(p)) for p in itertools.permutations(deps))


project = {"main": ["pupils"], "pupils": ["config", "marks"], "marks": ["config"], "config": []}
assert load_order(project) == ["config", "marks", "pupils", "main"], "пример из условия"
assert project == {"main": ["pupils"], "pupils": ["config", "marks"], "marks": ["config"], "config": []}, \
    "словарь менять нельзя"
assert load_order({}) == []
assert load_order({"a": []}) == ["a"]
error = error_of(load_order, {"main": ["pupils"], "pupils": ["marks"], "marks": ["pupils"]})
assert isinstance(error, ImportError) and str(error).startswith("циклический импорт"), \
    "круг — ImportError с сообщением «циклический импорт: ...»"
error = error_of(load_order, {"a": ["a"]})
assert isinstance(error, ImportError), "модуль, который импортирует сам себя, — тоже круг"

rnd = random.Random(11)
for _ in range(300):
    n = rnd.randint(1, 5)
    names = [f"m{i}" for i in range(n)]
    deps = {name: [other for other in names if other != name and rnd.random() < 0.3] for name in names}
    if rnd.random() < 0.2:
        name = rnd.choice(names)
        deps[name].append(name)
    copy = {name: list(value) for name, value in deps.items()}
    if has_order(deps):
        order = load_order(deps)
        assert check_order(deps, order), (deps, order)
    else:
        error = error_of(load_order, deps)
        assert isinstance(error, ImportError) and str(error).startswith("циклический импорт"), deps
    assert deps == copy, "словарь менять нельзя"

big = {f"m{i}": [f"m{i + 1}"] for i in range(300)}
big["m300"] = []
assert check_order(big, load_order(big)), "длинная цепочка импортов"
print("Задача 11: ок")

### Задача 12. Свой `import` ★

Напишите `load(name, folders, loaded)` — упрощённый `import`.

- `loaded` — словарь «имя → модуль», как `sys.modules`. Если `name` уже есть в нём, верните готовый
  модуль: файл второй раз не выполняется.
- Иначе ищите файл `name.py` в папках из списка `folders` по порядку — как по `sys.path`. Первый найденный
  файл и становится модулем. Папки могут быть заданы и строками, и `Path`.
- Создайте пустой модуль `types.ModuleType(name)`, запишите в его `__file__` путь к файлу строкой
  и положите модуль в `loaded` — **до** выполнения файла. Потом выполните текст файла в пространстве имён
  модуля и верните модуль.
- Если файла нет ни в одной папке — `ModuleNotFoundError` с сообщением `нет модуля <имя>`.

```
make_project({"one/greet.py": "WHO = 'первая'", "two/greet.py": "WHO = 'вторая'"})
loaded = {}
greet = load("greet", [ROOT / "one", ROOT / "two"], loaded)
greet.WHO                                               ->  'первая'
load("greet", [ROOT / "one", ROOT / "two"], loaded) is greet   ->  True
```

*Подсказка:* `exec(text, namespace)` выполняет строку с кодом так, будто это файл, а созданные имена
кладёт в словарь `namespace`. У модуля такой словарь — `module.__dict__`: имена из него и есть атрибуты
модуля. Файл читает `Path(...).read_text(encoding="utf-8")`, есть ли он — `.exists()`.

In [ ]:
import types
from pathlib import Path


def load(name: str, folders: list, loaded: dict):
    """Упрощённый import: ищет name.py в папках folders, запоминает модуль в loaded."""
    if name in loaded:
        return loaded[name]
    for folder in folders:
        path = Path(folder) / f"{name}.py"
        if path.exists():
            module = types.ModuleType(name)
            module.__file__ = str(path)
            loaded[name] = module                     # до выполнения: так import переживает круги
            exec(path.read_text(encoding="utf-8"), module.__dict__)
            return module
    raise ModuleNotFoundError(f"нет модуля {name}")




make_project({
    "one/greet.py": "WHO = 'первая'\n",
    "two/greet.py": "WHO = 'вторая'\n",
    "two/calc.py": "RUNS = []\nRUNS.append(1)\n\n\ndef double(x):\n    return 2 * x\n",
    "two/selfcheck.py": "REGISTERED = _probe()\n",
})
folders = [ROOT / "one", ROOT / "two"]
loaded = {}
greet = load("greet", folders, loaded)
assert isinstance(greet, types.ModuleType), "load возвращает модуль: types.ModuleType(name)"
assert greet.WHO == "первая", "папки просматриваются по порядку, первая находка — модуль"
assert greet.__name__ == "greet" and Path(greet.__file__) == ROOT / "one" / "greet.py", \
    "у модуля имя name и __file__ — путь к найденному файлу"
assert loaded == {"greet": greet}, "модуль запоминается в loaded под своим именем"
assert load("greet", folders, loaded) is greet, "второй раз модуль берётся из loaded"
calc = load("calc", [str(folder) for folder in folders], loaded)
assert calc.double(21) == 42, "папки могут быть и строками"
assert load("calc", folders, loaded) is calc and calc.RUNS == [1], "файл выполняется один раз"
assert load("greet", [ROOT / "two"], {}).WHO == "вторая", "у другого loaded — своя загрузка"

probe_loaded = {}
builtins._probe = lambda: "selfcheck" in probe_loaded     # файл спросит, лежит ли он уже в loaded
try:
    selfcheck = load("selfcheck", folders, probe_loaded)
finally:
    del builtins._probe
assert selfcheck.REGISTERED is True, \
    "модуль кладут в loaded до выполнения файла — так import переживает круговые импорты"

error = error_of(load, "nothing", folders, loaded)
assert isinstance(error, ModuleNotFoundError) and str(error) == "нет модуля nothing", \
    "нет файла — ModuleNotFoundError: нет модуля nothing"
assert set(loaded) == {"greet", "calc"}, "ненайденный модуль в loaded не попадает"
print("Задача 12: ок")

## Что дальше

Дальше в разделе 3 — стиль кода: соглашение PEP 8, программы, которые проверяют и сами исправляют
оформление, аннотации типов и докстроки. Пригодится всё, что сегодня разложили по модулям.